# Phase 4 - Object Understanding with Qwen2.5-VL

This notebook runs Phase 4 on Kaggle using GPU. It reads representative crops from Phase 3 and produces structured metadata for later phases.

In [ ]:
!pip install -q torch torchvision transformers accelerate pillow tqdm sentencepiece einops peft
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate

In [ ]:
import os
import json
import sys
from pathlib import Path

ROOT = '/kaggle/working'
os.chdir(ROOT)

if os.path.exists('/kaggle/input'):
    print('Kaggle input mounted')

# Clone the project if needed
project_dir = Path('/kaggle/working/AI_Video_Intelligence')
if not project_dir.exists():
    !git clone https://github.com/your-user/AI_Video_Intelligence.git /kaggle/working/AI_Video_Intelligence
    os.chdir(project_dir)
else:
    os.chdir(project_dir)

sys.path.insert(0, str(project_dir))
print('Working directory:', project_dir)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))

In [ ]:
from ai.pipeline.phase4_pipeline import Phase4Pipeline

# Adjust these paths if you upload the project differently
representative_crop_dir = 'outputs/phase3/production_runs/test/04_representative_selection/crops'
output_dir = 'outputs/phase4'

pipeline = Phase4Pipeline(
    representative_crop_dir=representative_crop_dir,
    output_dir=output_dir,
    batch_size=8,
)

pipeline.run()

## Verify outputs

Check that the expected files were generated with timing information.

In [ ]:
from pathlib import Path
import json

output_dir = Path('outputs/phase4')
for path in [
    output_dir / 'track_metadata.json',
    output_dir / 'object_metadata.json',
    output_dir / 'event_metadata.json',
    output_dir / 'semantic_metadata.json',
]:
    print(path, path.exists())

if (output_dir / 'track_metadata.json').exists():
    with open(output_dir / 'track_metadata.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    if data:
        first = data[0]
        print('sample keys:', sorted(first.keys()))
        for key in ['start_time_seconds', 'end_time_seconds', 'duration_seconds', 'start_timestamp', 'end_timestamp', 'timestamp']:
            print(key, first.get(key))